In [1]:
import torch
from torch_scatter import scatter_mean
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D
import networkx as nx
import os
import pickle
import sys
sys.path.append(os.path.dirname(os.path.dirname(os.getcwd())))

from methods import mcmc_community_delays, mmca_community_delays
from methods.utils import graph_dynamic_delays

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [2]:
# ---------------初始数据-----------------------------

# 导入默认参数
file_path = os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), 'parameters/community_data_1.pkl')
with open(file_path, 'rb') as f:
    init_data = pickle.load(f)

epi_paras = torch.tensor(init_data['epi_paras'], dtype=torch.float32).to(device)
soc_paras = torch.tensor(init_data['soc_paras'], dtype=torch.float32).to(device)
features_state = torch.tensor(init_data['init_state'], dtype=torch.float32).to(device)
P_rows, P_cols = init_data['P_matrix'].nonzero()
P_edge_index = torch.tensor(np.array([P_rows, P_cols]), dtype=torch.long).to(device)
communities = init_data['communities']
P_community = init_data['P_community']
P_G = init_data['P_G']

I_rows, I_cols = init_data['I_matrix'].nonzero()
I_edge_index = torch.tensor(np.array([I_rows, I_cols]), dtype=torch.long).to(device)

# # # ---------------初始数据-----------------------------
node_num =  features_state.shape[0]
time_scale = 1000

In [3]:
para_len = 100
soc_paras_mcmc = soc_paras.unsqueeze(0).repeat(para_len, 1)
epi_paras_mcmc = epi_paras.unsqueeze(0).repeat(para_len, 1, 1) 
features_state_tensor_mcmc = features_state.unsqueeze(0).repeat(para_len, 1, 1).to(device)
delays = 0
# sigmoid = None  # sigmoid参数
sigmoid = [10,10,5]  # sigmoid参数
mcmc = mcmc_community_delays.MCMC(para_len, device)
mcmc_features_times, _, mcmc_soc_attention = graph_dynamic_delays(time_scale, mcmc, features_state_tensor_mcmc.clone(), epi_paras_mcmc,soc_paras_mcmc, P_edge_index, I_edge_index, communities, device, delays,  sigmoid = sigmoid)
# torch.save(mcmc_features_times, f'./data/mcmc_features_times_{sigmoid[0]}_{sigmoid[1]}_{sigmoid[2]}.pt')
# torch.save(mcmc_soc_attention, f'./data/mcmc_soc_attention_{sigmoid[0]}_{sigmoid[1]}_{sigmoid[2]}.pt')

# torch.save(mcmc_features_times, f'./data/mcmc_features_times_{sigmoid[0]}_X_{sigmoid[2]}.pt')
# torch.save(mcmc_soc_attention, f'./data/mcmc_soc_attention_{sigmoid[0]}_X_{sigmoid[2]}.pt')

torch.save(mcmc_features_times, f'./data/mcmc_features_times_{sigmoid[0]}_{sigmoid[1]}_X.pt')
torch.save(mcmc_soc_attention, f'./data/mcmc_soc_attention_{sigmoid[0]}_{sigmoid[1]}_X.pt')

# torch.save(mcmc_features_times, f'./data/mcmc_features_times_{sigmoid[0]}_X_X.pt')
# torch.save(mcmc_soc_attention, f'./data/mcmc_soc_attention_{sigmoid[0]}_X_X.pt')

e:\博士期间论文\信息物理社会项目\三层网络动力学\project0430\codes\Dynamics\methods\utils.py:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  center = torch.tensor(center, dtype=x.dtype, device=x.device)
e:\博士期间论文\信息物理社会项目\三层网络动力学\project0430\codes\Dynamics\methods\utils.py:30: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x0 = torch.tensor(x0, dtype=x.dtype, device=x.device)
e:\博士期间论文\信息物理社会项目\三层网络动力学\project0430\codes\Dynamics\methods\mcmc_community_delays.py:22: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  communities = torch.tensor(communities, device=se